# PyTorch 学习教程

这是一个交互式的 PyTorch 学习教程，你可以边学习边运行代码。

## 目录
1. [环境检查与安装](#1-环境检查与安装)
2. [张量基础](#2-张量基础)
3. [自动微分](#3-自动微分)
4. [构建神经网络](#4-构建神经网络)
5. [训练模型](#5-训练模型)
6. [实战：MNIST 手写数字识别](#6-实战mnist-手写数字识别)

## 1. 环境检查与安装

首先检查 PyTorch 是否已安装，以及 CUDA 是否可用。

In [ ]:
# 安装 PyTorch (如果还没安装，取消下面的注释)
# !pip install torch torchvision torchaudio

In [5]:
import torch
import numpy as np

print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 是否可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA 版本: {torch.version.cuda}')
    print(f'GPU 设备: {torch.cuda.get_device_name(0)}')
print("gpu 设备：", torch.device)

PyTorch 版本: 2.8.0
CUDA 是否可用: False
gpu 设备： <class 'torch.device'>


## 2. 张量基础

张量（Tensor）是 PyTorch 中最基本的数据结构，类似于 NumPy 的数组，但可以在 GPU 上运行。

### 2.1 创建张量

In [ ]:
import torch

# 从列表创建
tensor_from_list = torch.tensor([1, 2, 3, 4])
print('从列表创建:', tensor_from_list)

# 创建特殊张量
zeros = torch.zeros(3, 4)  # 全零张量
ones = torch.ones(2, 3)  # 全一张量
random = torch.rand(2, 3)  # 随机张量 [0, 1)
randn = torch.randn(2, 3)  # 标准正态分布

print('\n全零张量:')
print(zeros)
print('\n随机张量:')
print(random)

### 2.2 张量操作

In [ ]:
# 基本运算
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

c = a + b  # 加法
d = a * b  # 逐元素乘法
e = a @ b  # 点积

print('a + b =', c)
print('a * b =', d)
print('a @ b =', e)

# 形状操作
x = torch.randn(2, 3, 4)
print('\n原始形状:', x.shape)
y = x.view(2, 12)  # 重塑
print('重塑后:', y.shape)
z = x.reshape(6, 4)  # 重塑（更灵活）
print('再次重塑:', z.shape)

## 3. 自动微分

PyTorch 的自动微分系统（autograd）是其核心功能之一，可以自动计算梯度。

In [19]:
# 创建需要梯度的张量
x = torch.tensor([2.0], requires_grad=True)
y = torch.tensor([3.0], requires_grad=True)

# 前向传播
z = x ** 2 + y ** 3
print('z =', z)

# 反向传播
z.backward()

# 查看梯度
print(f'dz/dx = {x.grad}  (应该是 2*x = 4.0)')
print(f'dz/dy = {y.grad}  (应该是 3*y^2 = 27.0)')

z = tensor([31.], grad_fn=<AddBackward0>)
dz/dx = tensor([4.])  (应该是 2*x = 4.0)
dz/dy = tensor([27.])  (应该是 3*y^2 = 27.0)


### 3.1 梯度管理

在训练过程中，我们需要管理梯度的计算和清零。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 第一次计算
y = x ** 2
z = y.sum()
z.backward()
print('第一次梯度:', x.grad)

# 清零梯度（重要！）
x.grad.zero_()

# 第二次计算
y = x ** 3
z = y.sum()
z.backward()
print('第二次梯度:', x.grad)

# 禁用梯度计算（推理时使用）
with torch.no_grad():
    y = x * 2
    print('\n禁用梯度时，y.requires_grad =', y.requires_grad)

## 4. 构建神经网络

使用 `nn.Module` 类来定义神经网络模型。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# 创建模型
model = SimpleNet(784, 128, 10)
print(model)

# 查看参数
print('\n模型参数:')
for name, param in model.named_parameters():
    print(f'{name}: {param.shape}')

### 4.1 卷积神经网络示例

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


cnn_model = CNN()
print(cnn_model)

# 测试输入
test_input = torch.randn(1, 1, 28, 28)  # batch_size=1, channels=1, height=28, width=28
output = cnn_model(test_input)
print(f'\n输入形状: {test_input.shape}')
print(f'输出形状: {output.shape}')

## 5. 训练模型

完整的训练流程包括：数据准备、前向传播、计算损失、反向传播、更新参数。

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. 准备模拟数据
X_train = torch.randn(1000, 784)
y_train = torch.randint(0, 10, (1000,))
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 2. 定义模型
model = SimpleNet(784, 128, 10)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'使用设备: {device}')

# 3. 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 训练循环
num_epochs = 5
for epoch in range(num_epochs):
    model.train()  # 设置为训练模式
    running_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        # 前向传播
        output = model(data)
        loss = criterion(output, target)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

print('\n训练完成！')

## 6. 实战：MNIST 手写数字识别

使用真实的 MNIST 数据集训练一个手写数字识别模型。

In [ ]:
from torchvision import datasets, transforms

# 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 下载并加载数据集
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f'训练集大小: {len(train_dataset)}')
print(f'测试集大小: {len(test_dataset)}')

In [ ]:
# 定义 MNIST 模型
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


mnist_model = MNISTNet().to(device)
print(mnist_model)

In [ ]:
# 训练函数
def train_epoch(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch}, Batch: {batch_idx}/{len(train_loader)}, Loss: {loss.item():.6f}')


# 测试函数
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest Loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')
    return accuracy

In [ ]:
# 开始训练
optimizer = optim.Adam(mnist_model.parameters(), lr=0.001)

print('开始训练 MNIST 模型...\n')
for epoch in range(1, 6):  # 训练 5 个 epoch
    train_epoch(mnist_model, device, train_loader, optimizer, epoch)
    test(mnist_model, device, test_loader)

print('训练完成！')

### 6.1 可视化预测结果

In [ ]:
import matplotlib.pyplot as plt

# 获取一些测试样本
examples = enumerate(test_loader)
batch_idx, (example_data, example_targets) = next(examples)

# 预测
with torch.no_grad():
    output = mnist_model(example_data.to(device))
    predictions = output.argmax(dim=1, keepdim=True)

# 可视化前 9 个样本
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(example_data[i][0], cmap='gray')
    ax.set_title(f'预测: {predictions[i].item()}, 真实: {example_targets[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 总结

通过本教程，你已经学习了：

1. **张量操作**：创建、操作和变换张量
2. **自动微分**：使用 autograd 自动计算梯度
3. **神经网络**：使用 nn.Module 构建模型
4. **训练流程**：完整的训练和评估流程
5. **实战应用**：MNIST 手写数字识别

### 下一步学习

- 尝试不同的网络架构（ResNet、VGG 等）
- 学习迁移学习和预训练模型
- 探索其他数据集（CIFAR-10、ImageNet）
- 了解更高级的技术（注意力机制、Transformer）

### 练习建议

1. 修改网络结构，观察对性能的影响
2. 尝试不同的优化器和学习率
3. 添加数据增强技术
4. 实现早停法和学习率调度
5. 保存和加载训练好的模型

祝你学习愉快！🚀